# Week 1 - Day 2: Your First Agent (instructor solution)

Today we go from a plain LLM call to a working agent, one small step at a time.

| Part | What we do |
|---|---|
| **1. Basic LLM** | Call a model, give it a short-term memory, and see what it cannot do |
| **2. One tool** | Write a web search tool and use it by hand |
| **3. Workflow** | Fixed steps that we write: search words, search, answer |
| **4. Function calling** | See the model ask for the tool, and how the answer comes back |
| **5. Agent** | Let the model decide the steps, with a proper system prompt and a short-term memory |
| **6. Your turn** | You write your own tool and give it to the agent |

Everything is free: **Groq** for the model and **DuckDuckGo** for the web search. You need one free key (Groq). The search needs no key.

## Setup
1. Get a free **Groq** key at **console.groq.com** -> API Keys.
2. Give the notebook your key:
   - **On Google Colab:** click the Secrets icon (key) in the left bar, add a secret named `GROQ_API_KEY`, paste your key as the value, and switch **Notebook access** on.
   - **On your own computer:** put `GROQ_API_KEY=your_key_here` in a file called `.env` in this folder (or any folder above it).

The web search (DuckDuckGo) needs no key. Your key is like a password: never share it and never type it into the notebook.

**Optional:** if you have a free **LangSmith** key (smith.langchain.com), add it the same way with the name `LANGSMITH_API_KEY`. Every run is then recorded online, so you can open it and see each model call, the tool that was called, its inputs and result, the time and the tokens. Without the key, nothing changes.

In [ ]:
%pip install -q langchain langchain-groq ddgs python-dotenv

In [ ]:
import os
import time
from datetime import date
from dotenv import find_dotenv, load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import InMemorySaver
from ddgs import DDGS

load_dotenv(find_dotenv(usecwd=True), override=True)    # on your computer: reads your .env file
GROQ_API_KEY = os.getenv("GROQ_API_KEY", "")

try:                                                     # on Google Colab: reads the key from Secrets
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY") or GROQ_API_KEY
except Exception:
    pass

if GROQ_API_KEY == "":
    raise RuntimeError("No GROQ_API_KEY found. Add it (Colab: Secrets, computer: .env file), then run this cell again.")

# Optional: record every run in LangSmith (smith.langchain.com). Skipped when there is no key.
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY", "")
try:
    from google.colab import userdata
    LANGSMITH_API_KEY = userdata.get("LANGSMITH_API_KEY") or LANGSMITH_API_KEY
except Exception:
    pass

if LANGSMITH_API_KEY != "":
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
    os.environ["LANGSMITH_PROJECT"] = "day2-first-agent"
    print("LangSmith tracing is ON. Open smith.langchain.com and look for the project day2-first-agent.")

MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-120b")
THINKING = "low" if MODEL.startswith("openai/gpt-oss") else None    # "low" makes gpt-oss models answer faster

llm = ChatGroq(model=MODEL, api_key=GROQ_API_KEY, temperature=0.3, reasoning_effort=THINKING)
today = str(date.today())          # a model does not know today's date, so we will tell it
print("Ready. Model:", MODEL)

---
# Part 1 - Basic LLM

One question in, one answer out. `llm.invoke(...)` sends the question to the model and returns a **message object**. The text is in `.content`.

In [ ]:
# Ask the model one question. llm.invoke(...) sends it and gives back a message object.
reply = llm.invoke("Say hello to a class of AI students in one short sentence.")

# The text of the answer is in reply.content
print(reply.content)

### What is inside the reply?
The reply is more than text. It also says how many **tokens** were used.

In [ ]:
# The reply is more than text. Let us look inside it.
print("Type  :", type(reply).__name__)       # what kind of object it is (an AIMessage)
print("Text  :", reply.content)               # the answer text
print("Tokens:", reply.usage_metadata)        # how many tokens this call used (input, output, total)

### Messages and roles
A chat has three kinds of message:

| Role | Class | Who writes it |
|---|---|---|
| system | `SystemMessage` | You: the rules for the model |
| user | `HumanMessage` | The person asking |
| assistant | `AIMessage` | The model's earlier answers |

We send a **list** of messages. The system message changes how the model behaves.

In [ ]:
# Send a LIST of messages instead of one string.
# The SystemMessage sets the rules for the model. The HumanMessage is the question.
reply = llm.invoke([
    SystemMessage("You are a pirate. Answer in one short sentence."),
    HumanMessage("What is an AI agent?"),
])

print(reply.content)

### Short-term memory: a model forgets
A model remembers **nothing** between calls. Every `llm.invoke(...)` starts fresh. Watch: we tell it our name, then ask for it in a **second, separate call**.

In [ ]:
# Call 1: tell the model our name.
reply = llm.invoke("My name is Aarav. Say hi in five words.")
print(reply.content)

# Call 2: ask for the name. This is a NEW call, so the model has no memory of call 1.
reply = llm.invoke("What is my name? Answer in one short sentence.")
print(reply.content)

The second call is brand new, so the model cannot know. A chat app seems to remember because it sends the **whole conversation again** every time. We can do the same: keep a **list of messages**, add every question and every answer to it, and send the whole list.

In [ ]:
# Our own memory: a list that holds the whole conversation.
chat = []

# Turn 1: add our message, send the whole list, then add the model's answer to the list.
chat.append(HumanMessage("My name is Aarav. Say hi in five words."))
reply = llm.invoke(chat)
chat.append(reply)                      # remember the answer too
print(reply.content)

# Turn 2: the list now holds turn 1, so the model can answer from it.
chat.append(HumanMessage("What is my name? Answer in one short sentence."))
reply = llm.invoke(chat)                # the model now sees the whole list
chat.append(reply)
print(reply.content)

# 4 messages: question 1, answer 1, question 2, answer 2.
print("Messages kept:", len(chat))

The model did not remember anything: **we** sent the earlier messages again. That list is the conversation's **state**, and keeping it is **short-term memory**. It lasts only while the list exists, and a longer list means more tokens to send each time.

### What a plain LLM cannot do
A model only knows what it learned during training. It has **no internet**, so it cannot look anything up, and it cannot know what happened today. Read the answer: did it search, or did it guess?

In [ ]:
# A model has no internet. Ask it to search and see what it does: refuse, or guess?
reply = llm.invoke("Search the web and tell me today's top news headline from Nepal.")

print(reply.content)

It cannot search. It either refuses or invents a headline from memory. To look things up, the model needs a **tool**. Let us write one.

---
# Part 2 - One tool

A **tool** is a normal Python function with `@tool` on top. The model never sees the code. It only sees three things: the **name**, the **docstring** (the sentence in triple quotes) and the **inputs**.

Our tool searches the web with DuckDuckGo. It returns the top three results: a title, a short summary and a link.

In [ ]:
@tool                                     # @tool turns this function into a tool the model can use
def search_web(query: str) -> str:
    """Search the web and return the top results with a short summary and a link for each."""   # the model reads this sentence
    # Step 1: ask DuckDuckGo for 3 results. If it fails, wait a moment and try once more.
    results = None
    for attempt in range(2):                      # the free search sometimes says "slow down", so we try twice
        try:
            results = DDGS().text(query, max_results=3)
            break                                 # it worked, so stop trying
        except Exception:
            time.sleep(2)                         # it failed, so wait 2 seconds

    # Step 2: if both tries failed, return a short message. A good tool never crashes.
    if results is None:
        return "Error: the search failed. Check the search words, or try again in a moment."

    # Step 3: the search worked but found nothing.
    if len(results) == 0:
        return "Error: nothing found for " + query + ". Try different words."

    # Step 4: turn each result into one line of text: title, short summary (first 300 characters), link.
    lines = []
    for result in results:
        lines.append(result["title"] + ": " + result["body"][:300] + " (" + result["href"] + ")")
    return "\n".join(lines)               # join the lines into one text, one result per line

### What the model sees
`@tool` turned our function into a tool object. These three lines are everything the model knows about it.

In [ ]:
# These three things are ALL the model knows about the tool.
print("Name       :", search_web.name)
print("Description:", search_web.description)    # comes from the docstring
print("Inputs     :", search_web.args)           # comes from the function's inputs

### Using the tool by hand
We can call a tool ourselves with `.invoke(...)`. No model is involved.

In [ ]:
# Call the tool by hand with .invoke(...). No model is involved.
print(search_web.invoke({"query": "Phewa Lake Pokhara"}))

### When the search fails
A good tool does not crash. It **returns a short message starting with `Error:`** and the model can read it and react. Here we search for nothing (an empty query) to see it.

In [ ]:
# An empty search fails. The tool returns an "Error:" message instead of crashing.
print(search_web.invoke({"query": ""}))

---
# Part 3 - Workflow

Now we use the tool inside a **workflow**. **We** write the steps and their order. The model does the thinking inside two of them, and our code runs the search.

```
question  ->  [ LLM: write search words ]  ->  [ code: search_web ]  ->  [ LLM: write the answer ]
```

In [ ]:
# A WORKFLOW: three steps, in an order that WE decided.
question = "What is the latest news about tourism in Nepal?"

# Step 1 (LLM): turn the question into good search words
search_words = llm.invoke("Write the best web search query for this question, in 3 to 6 plain words. Use no quotes, no operators and no years. Today is " + today + ". Reply with ONLY the query. Question: " + question).content
print("Step 1 - search words:", search_words)

# Step 2 (code): run the search
results = search_web.invoke({"query": search_words})
print("Step 2 - search results:", results[:200])     # [:200] = show only the first 200 characters

# Step 3 (LLM): write the answer from the results
answer = llm.invoke("Answer the question using only these search results. If the results do not contain the answer, say so. Answer in two or three short sentences and end with a link.\n\nQuestion: " + question + "\n\nResults:\n" + results).content
print("Step 3 - answer:", answer)

The order never changes: search words, search, answer. That makes it **predictable** and easy to test.

### It breaks when the question changes
This question needs **two** searches, one for each phone maker. Our workflow has only one search step, because we wrote it that way. Run the same three steps and read the answer.

In [ ]:
# The same three steps, for a question that needs TWO searches (one for Apple, one for Samsung).
question = "What are the newest flagship phones from Apple and from Samsung, and how much do they cost?"

# Step 1 (LLM): turn the question into good search words
search_words = llm.invoke("Write the best web search query for this question, in 3 to 6 plain words. Use no quotes, no operators and no years. Today is " + today + ". Reply with ONLY the query. Question: " + question).content
print("Step 1 - search words:", search_words)

# Step 2 (code): run the search. There is only ONE search step, because we wrote it that way.
results = search_web.invoke({"query": search_words})
print("Step 2 - search results:", results[:200])

# Step 3 (LLM): write the answer from the results
answer = llm.invoke("Answer the question using only these search results. If the results do not contain the answer, say so. Answer in two or three short sentences and end with a link.\n\nQuestion: " + question + "\n\nResults:\n" + results).content
print("Step 3 - answer:", answer)

One search gave results about one topic (or a mix), so the answer is probably incomplete. To handle a question like this, **someone has to write another step**.

**Takeaway:** a workflow is reliable when you know the steps in advance. When you do not, you need something that can **decide** the steps. That is an agent, and we get there after seeing how a model uses a tool.

---
# Part 4 - Function calling

How does the model use a tool? It does **not** run it. It only **asks**. Our code runs the tool and hands the result back. This is called **function calling**, and it has three steps:

1. **Ask:** the model is told about the tool, and replies with a request.
2. **Run:** our code runs the tool with the inputs the model chose.
3. **Answer:** we give the result back, and the model writes the final answer.

### Step 1 - The model asks
`bind_tools` tells the model which tools exist. We also send a short **system message** with today's date, because a model does not know it. Look at what comes back: no answer, only a request.

In [ ]:
# Tell the model which tools exist. This does not run anything.
llm_with_tools = llm.bind_tools([search_web])

# A system message: today's date (a model does not know it) and how to answer.
system_message = SystemMessage("Today's date is " + today + ". Do not add a year to your search words unless the user gave one. "
                               "When you answer, use only the tool result, in two or three short sentences, and end with the link of the page you used.")
question = "What is the latest news about tourism in Nepal?"
reply = llm_with_tools.invoke([system_message, HumanMessage(question)])

# Step 1 - the model does NOT answer. It ASKS for a tool: the text is empty and tool_calls holds the request.
print("Text      :", reply.content)
print("Tool calls:", reply.tool_calls)

The text is empty. `tool_calls` holds the request: the tool **name** and the **inputs** the model chose by reading the docstring. Nothing has run yet.

### Step 2 - We run the tool
We use the name and the inputs from the request.

In [ ]:
# Step 2 - WE run the tool. The request is a dictionary with the tool name, its inputs and an id.
call = reply.tool_calls[0]
print("The model asked for:", call["name"], call["args"])

# Run the tool with the inputs the model chose.
search_result = search_web.invoke(call["args"])
print("Our tool returned  :", search_result[:300])

### Step 3 - We give the result back
We send the whole conversation: the same system message, the question, the model's request, and a `ToolMessage` with the result. The system message tells the model to answer **only from the tool result**. Without it, a model may add facts it made up. Now the model can write the answer.

In [ ]:
# Step 3 - the whole conversation so far, in order:
messages = [
    system_message,                                                      # the rules
    HumanMessage(question),                                              # the question
    reply,                                                               # the model's request for the tool
    ToolMessage(content=search_result, tool_call_id=call["id"]),         # the tool's result, linked to the request by its id
]

# Ask the model again. Now it has the result, so it can write the final answer.
final = llm_with_tools.invoke(messages)
print(final.content)

That is the whole idea: **ask, run, answer**. An agent does exactly these steps, in a loop, by itself.

---
# Part 5 - Agent

```
question  ->  [ LLM decides ]  <->  [ tools ]  ->  answer
```
An **agent** is a model plus tools plus a loop. `create_agent` builds all of it. We give it three things:

| Input | What it is |
|---|---|
| the model | the brain: `llm` |
| `tools` | what it is allowed to do |
| `system_prompt` | its job description |

First an agent with **no tools** (an empty list), to see the shape of the thing.

In [ ]:
# An agent with NO tools (an empty list). It is only a model with a system prompt.
agent = create_agent(llm, tools=[], system_prompt="You are a helpful assistant. Answer in one or two short sentences.")

# agent.invoke takes a dictionary with the messages. ("user", "...") is a short way to write a human message.
result = agent.invoke({"messages": [("user", "Search the web and tell me today's top news headline from Nepal.")]})

# result["messages"] is the whole conversation. The last one is the answer.
print(result["messages"][-1].content)

With no tools, the agent is only a model: it answers once, and it still cannot search. The result holds the whole conversation as a list of messages:

In [ ]:
# Print every message of the run: its type (human, ai, tool) and the first 100 characters.
for message in result["messages"]:
    print(message.type, "->", message.content[:100])

### Now we add the tool
We build the agent again, this time with `search_web`. We also write a proper **system prompt**: who the agent is, when to use the tool, and what to do when something goes wrong.

In [ ]:
# The job description of the agent. It says who it is, what it may do, and how to answer.
system_prompt = ("You are a helpful research assistant. Today's date is " + today + ". "
                 "Use the search_web tool to look facts up and never guess. "
                 "Do not add a year to your search words unless the user gave one. "
                 "Your only tool is search_web. You cannot open web pages, so use only the search results. "
                 "Write the link as plain text, with no citation markers. "
                 "Base your answer only on what the tool returns, and end with the link of the page you used. "
                 "If the tool returns an Error, tell the user and ask for different words. "
                 "Answer in two or three short sentences.")

# Build the agent: the model, the tools it may use, and the job description.
agent = create_agent(llm, tools=[search_web], system_prompt=system_prompt)

question = "What is the latest news about tourism in Nepal?"

print("> Entering agent...")
try:
    # agent.stream gives us one update for each step the agent takes.
    # recursion_limit=10 is the safety net: the agent stops after 10 steps.
    for update in agent.stream({"messages": [("user", question)]}, config={"recursion_limit": 10}, stream_mode="updates"):
        for node, data in update.items():             # node is "model" or "tools"
            message = data["messages"][-1]            # the newest message in this update
            if node == "tools":
                print("Observation:", message.content[:150])        # a tool ran: show what it returned
            elif message.tool_calls:
                for call in message.tool_calls:                      # the model asked for a tool
                    print("Action:", call["name"])
                    print("Action Input:", call["args"])
            else:
                print("Final Answer:", message.content)             # no tool asked for: this is the answer
except Exception as error:
    print("Stopped early:", str(error)[:200])          # a model or network error stops the run, but never crashes the cell
print("> Finished.")

### Reading the steps
The agent printed its own loop, the same **ask, run, answer** we did by hand in Part 4:

| Line | Meaning |
|---|---|
| `Action` | Which **tool** the model chose |
| `Action Input` | The **inputs** it chose for that tool |
| `Observation` | What the tool **returned** |
| `Final Answer` | What the model wrote when it was done |

If the agent needs a second search, you will see a second `Action` and `Observation` before the final answer.

If you see `Stopped early`, the model made a mistake (for example, it asked for a tool it was not given) or Groq said "slow down". Run the cell again.

**With LangSmith tracing on**, the same run is also online. Open the project `day2-first-agent` at smith.langchain.com, click the newest run, and you will see the steps as a tree: `model`, `tools`, `model`. Click the `tools` step to see `search_web`, its inputs and what it returned.

### The question the workflow could not handle
Nobody wrote "search two times". The agent reads the question and decides how many searches it needs. `recursion_limit=10` in the loop is the safety net: it stops the agent after 10 steps, so it can never run forever.

In [ ]:
# The question the workflow could not handle. The agent decides how many searches it needs.
question = "What are the newest flagship phones from Apple and from Samsung, and how much do they cost?"

print("> Entering agent...")
try:
    for update in agent.stream({"messages": [("user", question)]}, config={"recursion_limit": 10}, stream_mode="updates"):
        for node, data in update.items():
            message = data["messages"][-1]
            if node == "tools":
                print("Observation:", message.content[:150])        # what a tool returned
            elif message.tool_calls:
                for call in message.tool_calls:                      # the model asked for a tool
                    print("Action:", call["name"])
                    print("Action Input:", call["args"])
            else:
                print("Final Answer:", message.content)             # the answer
except Exception as error:
    print("Stopped early:", str(error)[:200])
print("> Finished.")

### Give the agent a short-term memory
An agent can keep its messages between questions too. We give `create_agent` a **saver** (`InMemorySaver`), and every question carries a **thread id**: the name of the conversation. Same thread id, same memory.

In [ ]:
# Same agent, plus a saver. The saver keeps the messages of each conversation.
agent = create_agent(llm, tools=[search_web], system_prompt=system_prompt, checkpointer=InMemorySaver())

# config holds the settings for these runs:
#   thread_id       = the name of the conversation (the same name means the same memory)
#   recursion_limit = stop after 10 steps
config = {"configurable": {"thread_id": "class-1"}, "recursion_limit": 10}

try:
    # Question 1 in the thread "class-1"
    result = agent.invoke({"messages": [("user", "What is the capital of Nepal?")]}, config)
    print("Answer 1:", result["messages"][-1].content)

    # Question 2 in the SAME thread. "its" works because the agent still has question 1.
    result = agent.invoke({"messages": [("user", "What is its population?")]}, config)
    print("Answer 2:", result["messages"][-1].content)
    print("Messages kept in this thread:", len(result["messages"]))       # question 1 + tool + answer 1 + question 2 + ... all kept
except Exception as error:
    print("Stopped early:", str(error)[:200])

The second question says "its", and the agent knew what it meant, because it still had the first question in its messages. Now the **same question in a new thread**:

In [ ]:
# A NEW thread name means a new, empty memory.
other_thread = {"configurable": {"thread_id": "class-2"}, "recursion_limit": 10}

try:
    result = agent.invoke({"messages": [("user", "What is its population?")]}, other_thread)
    print("New thread:", result["messages"][-1].content)     # without the history, the agent cannot know what "its" means
except Exception as error:
    print("Stopped early:", str(error)[:200])

A new thread has no history, so the agent has to guess or ask. Memory belongs to a **thread**. `InMemorySaver` keeps it only while this notebook is running: restart the notebook and it is gone.

---
# Part 6 - Your turn

Now **you** code. There are four tasks. Each has a **TODO**: a gap for you to fill. Fill the gap, run the cell, then run the **check** cell under it. The check tells you `PASS` or `NOT YET` and gives a hint.

| Task | You will |
|---|---|
| **1** | Write a tool called `multiply` |
| **2** | Give the agent both tools and ask a question that needs both |
| **3** | Read the agent's messages and list the tools it used, in order |
| **4** | Create your own tool |

### Task 1 - Write the `multiply` tool
A tool needs three things: `@tool` on top, a **docstring** the model reads, and a return value that is **text**. Fill in the gaps.

In [ ]:
@tool                                                 # makes the function a tool
def multiply(a: float, b: float) -> str:
    """Multiply two numbers and return the result."""    # the model reads this to decide when to use the tool
    return str(a * b)                                 # tools return TEXT, so we turn the number into text

In [ ]:
# Check Task 1: call the tool by hand with 17 and 19.
answer = multiply.invoke({"a": 17, "b": 19})
print("Your tool returned:", answer)

if "TODO" in multiply.description:
    print("NOT YET: write the docstring. The model reads it to decide when to use the tool.")
elif answer in ["323", "323.0"]:
    print("PASS: multiply works. 17 times 19 is 323.")
else:
    print("NOT YET: 17 times 19 should give 323. Return the product as text, for example str(a * b).")

### Task 2 - Give the agent both tools
The agent below only has `search_web`. Add `multiply` to its list of tools, then run the cell. The answer should be about 26,500 (three times the height of Mount Everest in metres).

In [ ]:
# A job description that mentions both tools.
lab_prompt = ("You are a helpful research assistant. Today's date is " + today + ". "
              "Use search_web to look facts up and multiply for exact maths, and never guess. "
              "You cannot open web pages. Write links as plain text. Answer in two or three short sentences.")

# Task 2: the agent now has BOTH tools in its list.
agent = create_agent(llm, tools=[search_web, multiply], system_prompt=lab_prompt)

question = "Search the web for the height of Mount Everest in metres, then multiply it by 3."

result = {"messages": []}                     # start with an empty result, so the next tasks still work if the run fails
try:
    result = agent.invoke({"messages": [("user", question)]}, config={"recursion_limit": 12})
    print(result["messages"][-1].content)
except Exception as error:
    print("Stopped early:", str(error)[:200])

### Task 3 - Read the messages
The agent's result holds every message of the run, in order (this is the agent's **state**). A message of type `tool` is a tool result, and it has the tool's `name`. Fill in the gap to build a list of the tools the agent used, in order.

In [ ]:
# Task 3: go through every message of the run and collect the name of each tool result.
tools_called = []
for message in result["messages"]:
    if message.type == "tool":                # a message of type "tool" is a tool result
        tools_called.append(message.name)     # message.name is the name of the tool that ran

print("Tools called:", tools_called)

In [ ]:
# Check Task 3: the agent should have used both tools.
if len(tools_called) == 0:
    print("NOT YET: tools_called is empty. Finish the loop above, and make sure Task 2 ran.")
elif "search_web" in tools_called and "multiply" in tools_called:
    print("PASS: the agent used", tools_called)
else:
    print("NOT YET: the agent used", tools_called, "but it should use both search_web and multiply. Is multiply in the tools list in Task 2? Run Task 2 again.")

### Task 4 - Create your own tool
Write a tool of your own. Ideas: `celsius_to_fahrenheit(celsius: float)`, `word_count(text: str)`, `count_vowels(text: str)`.

Rules: `@tool` on top, simple inputs (`str` or `float`), a one-sentence docstring, and **return text**. Name your function `my_tool` so the check below can find it.

In [ ]:
# Task 4: an example of your own tool. It converts Celsius to Fahrenheit.
@tool
def my_tool(celsius: float) -> str:
    """Convert a temperature from Celsius to Fahrenheit."""
    return str(celsius * 9 / 5 + 32)          # the formula: Celsius times 9/5, plus 32

In [ ]:
# Check Task 4: show what the model would see of your tool.
try:
    print("Name       :", my_tool.name)
    print("Description:", my_tool.description)
    print("Inputs     :", my_tool.args)
    if "TODO" in my_tool.description or len(my_tool.description) < 10:
        print("NOT YET: write a clear one-sentence docstring.")
    else:
        print("PASS: you made a tool. Next, add it to an agent and ask a question that needs it.")
except NameError:
    print("NOT YET: write a tool and call the function my_tool.")

---
## What we learned
- A **basic LLM** answers from training only. It has no live data.
- A **workflow** runs steps that **we** wrote, in a fixed order. It is predictable, but it cannot handle a step nobody wrote.
- An **agent** is a model plus tools plus a loop, built with `create_agent`. **The model decides the steps.**
- A **tool** is a Python function with `@tool`. The model sees its name, docstring and inputs.
- **Function calling** is: the model **asks**, our code **runs**, the model **answers**.
- **Short-term memory** is the earlier messages, sent again. For an LLM we keep the list ourselves. For an agent we use a saver and a thread id.
- A good **system prompt** and **safe tools** (errors returned as text, like `Error: the search failed`) make an agent reliable.

## More ideas
1. **Change the system prompt.** Make the agent answer as a pirate. Does the tool still get used?
2. **Search settings.** Change `max_results=3` to 5 in the tool. Does the agent give a better answer? What does it cost in tokens?
3. **If a search fails.** The free search can say "slow down" when many people use it at once. Ask the same question twice in a row and read the `Error:` message if you get one.
4. **Use your own tool.** Add `my_tool` from Task 4 to an agent and ask a question that needs it.